# 06 — Softmax（数值稳定的逐行 Softmax）

## 背景

Softmax 是 Transformer 注意力机制的核心组件，也是分类任务的标准输出层。其 GPU 实现的难点在于：

1. **数值稳定性**：直接计算 $e^x$ 会溢出（float16 最大约 65504）。标准做法是减去行最大值：$\text{softmax}(x_j) = e^{x_j - \max_k x_k} / \sum_k e^{x_k - \max_k x_k}$
2. **多遍扫描**：数值稳定版需要先扫描一遍求最大值，再求指数和，再归一化，共三遍。
3. **FlashAttention 的核心思想**（Online Softmax）正是将三遍合并为可流式处理的形式。

## 问题定义

$$\text{MAX}[i] = \max_j A[i, j]$$
$$B[i,j] = \frac{e^{A[i,j] - \text{MAX}[i]}}{\sum_k e^{A[i,k] - \text{MAX}[i]}}$$

In [1]:
import tilelang
import tilelang.language as T
import triton
import triton.language as tl
import torch

tilelang.disable_cache()
print("TileLang:", tilelang.__version__)
print("Triton:  ", triton.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")

Loading tilelang libs from dev root: /root/tilelang/build


TileLang: 0.1.10+rocm.gitf791a9ca
Triton:   3.7.0
GPU: Radeon RX 7900 XTX


## PyTorch 参考实现

In [2]:
N, M = 4096, 16384
A = torch.randn(N, M, dtype=torch.float32, device="cuda")

def ref_softmax(A):
    return torch.softmax(A, dim=1)

B_ref = ref_softmax(A)
print(f"A: {A.shape} → B: {B_ref.shape}")
print(f"Row-sum check (should ≈ 1): B[0].sum() = {B_ref[0].sum().item():.6f}")

A: torch.Size([4096, 16384]) → B: torch.Size([4096, 16384])
Row-sum check (should ≈ 1): B[0].sum() = 1.000000


## Triton 实现

每行由一个 Triton program 处理，三遍扫描均用 Python 循环 + `tl.max`/`tl.sum` 完成。关键技术：
- `tl.max(chunk, 0)` 对 `[BLOCK_M]` 张量归约，返回 0-d 标量
- `tl.maximum(scalar_a, scalar_b)` 更新全局最大值
- `chunk / row_sum` 标量广播到 `[BLOCK_M]`

In [3]:
@triton.jit
def _softmax_kernel(X_ptr, Y_ptr, M, BLOCK_M: tl.constexpr):
    row  = tl.program_id(0)
    base = X_ptr + row * M

    # Pass 1: find row max
    row_max = tl.load(base).to(tl.float32)   # initialise with first element
    for start in range(0, M, BLOCK_M):
        cols  = start + tl.arange(0, BLOCK_M)
        chunk = tl.load(base + cols, mask=cols < M, other=float("-inf")).to(tl.float32)
        row_max = tl.maximum(row_max, tl.max(chunk, 0))

    # Pass 2: exp(x - max), write out, accumulate sum
    row_sum = 0.0
    for start in range(0, M, BLOCK_M):
        cols = start + tl.arange(0, BLOCK_M)
        mask = cols < M
        x = tl.load(base + cols, mask=mask, other=0.0).to(tl.float32)
        e = tl.exp(x - row_max)
        tl.store(Y_ptr + row * M + cols, e, mask=mask)
        row_sum = row_sum + tl.sum(e, 0)

    # Pass 3: normalise
    for start in range(0, M, BLOCK_M):
        cols = start + tl.arange(0, BLOCK_M)
        mask = cols < M
        y = tl.load(Y_ptr + row * M + cols, mask=mask)
        tl.store(Y_ptr + row * M + cols, y / row_sum, mask=mask)

BLOCK_M_TRI = 1024

def triton_softmax(A):
    B = torch.empty_like(A)
    _softmax_kernel[(A.shape[0],)](A, B, A.shape[1], BLOCK_M=BLOCK_M_TRI)
    return B

B_tri = triton_softmax(A)
ok = torch.allclose(B_ref, B_tri, atol=1e-4)
print("Triton correctness:", "✅ PASSED" if ok else
      f"❌ FAILED  max_diff={torch.max(torch.abs(B_ref - B_tri)).item():.6f}")

Triton correctness: ✅ PASSED


## TileLang 实现

TileLang 同样使用三遍扫描：`T.fill(mx, -inf)` 初始化、`T.reduce_max` 求最大值、`T.exp2` 配合 `log2_e` 实现快速 exp、`T.reduce_sum` 累加。

> **注意**：PyTorch 通过 **MIOpen**（厂商优化库）路由此内核，内部使用 online softmax，全局访存更少。TileLang 的 3-pass 显式内核明显优于 Triton，并接近 MIOpen 的结果。

**gfx1100 调参**：`threads=256, BLOCK_M=256` — `BM ≤ threads` 约束使 `T.copy` 生成向量化循环而非单次加载，8 个 wavefront/block 在 RDNA3 上有良好的 CU 占用率。


In [4]:
# ── GPU 自适应配置 ────────────────────────────────────────────────────────────
# gfx1100 (RX 7900 XTX): threads=256, BLOCK_M=256
#   约束：BM <= threads（T.copy 布局），threads=256 → 8 wavefront/CU
# gfx1201 (R9700): 同为 WMMA/wave32，32 CU；BM=256 <= threads=256 约束同样成立
#   T.Kernel 的 threads 参数必须是编译期常量——两个 arch 都使用 256，无需分支
arch = torch.cuda.get_device_properties(0).gcnArchName
BM = 256  # both gfx1100 and gfx1201

@tilelang.jit(pass_configs={
    tilelang.PassConfigKey.TL_DISABLE_WARP_SPECIALIZED: True,
    tilelang.PassConfigKey.TL_DISABLE_TMA_LOWER: True,
})
def tl_softmax(A, BLOCK_M: int):
    log2_e = 1.44269504
    N_, M_ = T.const("N, M")
    dtype = T.float32
    A: T.Tensor((N_, M_), dtype)
    B = T.empty((N_, M_), dtype)
    with T.Kernel(N_, threads=256) as row:
        A_l = T.alloc_fragment((1, BLOCK_M), dtype)
        B_l = T.alloc_fragment((1, BLOCK_M), dtype)
        mx  = T.alloc_fragment((1,), dtype)
        sm  = T.alloc_fragment((1,), dtype)
        # Pass 1: 求行最大值
        T.fill(mx, float("-inf"))
        for m_blk in T.Serial(M_ // BLOCK_M):
            T.copy(A[row, m_blk * BLOCK_M], A_l)
            T.reduce_max(A_l, mx, dim=1, clear=False)
        # Pass 2: exp(x - max)，累加 sum
        T.clear(sm)
        for m_blk in T.Serial(M_ // BLOCK_M):
            T.copy(A[row, m_blk * BLOCK_M], A_l)
            for j in T.Parallel(BLOCK_M):
                B_l[0, j] = T.exp2(log2_e * (A_l[0, j] - mx[0]))
            T.reduce_sum(B_l, sm, dim=1, clear=False)
            T.copy(B_l, B[row, m_blk * BLOCK_M])
        # Pass 3: 归一化
        for m_blk in T.Serial(M_ // BLOCK_M):
            T.copy(B[row, m_blk * BLOCK_M], B_l)
            for j in T.Parallel(BLOCK_M):
                B_l[0, j] = B_l[0, j] / sm[0]
            T.copy(B_l, B[row, m_blk * BLOCK_M])
    return B

k = tl_softmax.compile(N=N, M=M, BLOCK_M=BM)
B_tl = k(A.clone())
ok = torch.allclose(B_ref, B_tl, atol=5e-4)
print("TileLang 正确性:", "✅ PASSED" if ok else
      f"❌ FAILED  max_diff={torch.max(torch.abs(B_ref - B_tl)).item():.6f}")

2026-05-31 14:23:29  [TileLang:tilelang.jit.kernel:INFO] (kernel.py:130): TileLang begins to compile kernel `tl_softmax` with `out_idx=[-1]`


2026-05-31 14:23:30  [TileLang:tilelang.jit.kernel:INFO] (kernel.py:138): TileLang completes to compile kernel `tl_softmax`
TileLang 正确性: ✅ PASSED


## 性能对比

In [5]:
WARMUP, REPEAT = 20, 200

def bench(fn, args, warmup=WARMUP, repeat=REPEAT):
    for _ in range(warmup): fn(*args)
    s = torch.cuda.Event(enable_timing=True)
    e = torch.cuda.Event(enable_timing=True)
    torch.cuda.synchronize(); s.record()
    for _ in range(repeat): fn(*args)
    e.record(); torch.cuda.synchronize()
    return s.elapsed_time(e) / repeat

results = {
    "PyTorch":  bench(ref_softmax,    [A]),
    "Triton":   bench(triton_softmax, [A]),
    "TileLang": bench(k,             [A]),
}

bytes_rw = N * M * 4 * 2   # float32，读+写各一遍（3-pass 实际 5 遍，此处为有效 IO）
pt_ms  = results["PyTorch"]
tri_ms = results["Triton"]

header = f"{'实现':<12}  {'延迟 (ms)':>10}  {'带宽 (TB/s)':>12}  {'vs PyTorch':>10}  {'vs Triton':>10}"
sep    = "-" * len(header)
print(f"Softmax  (N={N}, M={M}, float32)")
print(sep)
print(header)
print(sep)
for name, ms in results.items():
    bw     = bytes_rw / ms / 1e9
    vs_pt  = f"{pt_ms/ms:+.1%}"
    vs_tri = f"{tri_ms/ms:+.1%}"
    print(f"{name:<12}  {ms:>10.4f}  {bw:>12.2f}  {vs_pt:>10}  {vs_tri:>10}")
print(sep)


Softmax  (N=4096, M=16384, float32)
--------------------------------------------------------------
实现               延迟 (ms)     带宽 (TB/s)  vs PyTorch   vs Triton
--------------------------------------------------------------
PyTorch           0.7188          0.75     +100.0%     +119.5%
Triton            0.8590          0.62      +83.7%     +100.0%
TileLang          1.1557          0.46      +62.2%      +74.3%
--------------------------------------------------------------


In [6]:

# ── GPU 自适应配置（Online Softmax）─────────────────────────────────────────────
# gfx1100 (RX 7900 XTX): threads=512, BLOCK_M=4096
# gfx1201 (R9700):       threads=512, BLOCK_M=4096
# gfx1151 (Radeon 8060S): threads=256, BLOCK_M=1024
#   之前因 tirx.exp2 在向量类型下触发 InternalError 而被限制在 BM=256。
#   修复方式：在 intrin_rule_hip.cc 中注册 tirx.* math ops 的 hip.FLowerIntrinsic。
#   BM=1024（16 个 tile）性能与 PyTorch/MIOpen 持平：2.557 ms vs 2.547 ms (+99.6%)。
arch = torch.cuda.get_device_properties(0).gcnArchName

# ── gfx1151 kernel（threads=256, BLOCK_M=1024）────────────────────────────────
@tilelang.jit(pass_configs={
    tilelang.PassConfigKey.TL_DISABLE_WARP_SPECIALIZED: True,
    tilelang.PassConfigKey.TL_DISABLE_TMA_LOWER: True,
})
def tl_softmax_online_gfx1151(A, BLOCK_M: int):
    log2_e = 1.44269504
    N_, M_ = T.const("N, M")
    dtype  = T.float32
    A: T.Tensor((N_, M_), dtype)
    B = T.empty((N_, M_), dtype)
    with T.Kernel(N_, threads=256) as row:
        Al    = T.alloc_fragment((1, BLOCK_M), dtype)
        mx    = T.alloc_fragment((1,), dtype)
        sm    = T.alloc_fragment((1,), dtype)
        mx_bl = T.alloc_fragment((1,), dtype)
        sm_bl = T.alloc_fragment((1,), dtype)
        T.fill(mx, float("-inf"))
        T.clear(sm)
        for m_blk in T.Serial(M_ // BLOCK_M):
            T.copy(A[row, m_blk * BLOCK_M], Al)
            T.fill(mx_bl, float("-inf"))
            T.reduce_max(Al, mx_bl, dim=1, clear=True)
            for i in T.Parallel(1):
                new_mx = T.max(mx[0], mx_bl[0])
                sm[0]  = sm[0] * T.exp2(log2_e * (mx[0] - new_mx))
                mx[0]  = new_mx
            for j in T.Parallel(BLOCK_M):
                Al[0, j] = T.exp2(log2_e * (Al[0, j] - mx[0]))
            T.clear(sm_bl)
            T.reduce_sum(Al, sm_bl, dim=1, clear=True)
            for i in T.Parallel(1):
                sm[0] = sm[0] + sm_bl[0]
        for m_blk in T.Serial(M_ // BLOCK_M):
            T.copy(A[row, m_blk * BLOCK_M], Al)
            for j in T.Parallel(BLOCK_M):
                B[row, m_blk * BLOCK_M + j] = T.exp2(log2_e * (Al[0, j] - mx[0])) / sm[0]
    return B

# ── gfx1100/gfx1201 kernel（threads=512, BLOCK_M=4096）────────────────────────
@tilelang.jit(pass_configs={
    tilelang.PassConfigKey.TL_DISABLE_WARP_SPECIALIZED: True,
    tilelang.PassConfigKey.TL_DISABLE_TMA_LOWER: True,
})
def tl_softmax_online_gfx11xx(A, BLOCK_M: int):
    log2_e = 1.44269504
    N_, M_ = T.const("N, M")
    dtype  = T.float32
    A: T.Tensor((N_, M_), dtype)
    B = T.empty((N_, M_), dtype)
    with T.Kernel(N_, threads=512) as row:
        Al    = T.alloc_fragment((1, BLOCK_M), dtype)
        mx    = T.alloc_fragment((1,), dtype)
        sm    = T.alloc_fragment((1,), dtype)
        mx_bl = T.alloc_fragment((1,), dtype)
        sm_bl = T.alloc_fragment((1,), dtype)
        T.fill(mx, float("-inf"))
        T.clear(sm)
        for m_blk in T.Serial(M_ // BLOCK_M):
            T.copy(A[row, m_blk * BLOCK_M], Al)
            T.fill(mx_bl, float("-inf"))
            T.reduce_max(Al, mx_bl, dim=1, clear=True)
            for i in T.Parallel(1):
                new_mx = T.max(mx[0], mx_bl[0])
                sm[0]  = sm[0] * T.exp2(log2_e * (mx[0] - new_mx))
                mx[0]  = new_mx
            for j in T.Parallel(BLOCK_M):
                Al[0, j] = T.exp2(log2_e * (Al[0, j] - mx[0]))
            T.clear(sm_bl)
            T.reduce_sum(Al, sm_bl, dim=1, clear=True)
            for i in T.Parallel(1):
                sm[0] = sm[0] + sm_bl[0]
        for m_blk in T.Serial(M_ // BLOCK_M):
            T.copy(A[row, m_blk * BLOCK_M], Al)
            for j in T.Parallel(BLOCK_M):
                B[row, m_blk * BLOCK_M + j] = T.exp2(log2_e * (Al[0, j] - mx[0])) / sm[0]
    return B

if arch.startswith("gfx1151"):
    BM_online = 1024   # 修复 tirx.exp2 HIP 注册缺失后从 256 提升
    k_online = tl_softmax_online_gfx1151.compile(N=N, M=M, BLOCK_M=BM_online)
else:
    BM_online = 4096
    k_online = tl_softmax_online_gfx11xx.compile(N=N, M=M, BLOCK_M=BM_online)

B_online = k_online(A.clone())
ok = torch.allclose(B_ref, B_online, atol=5e-4)
print("TileLang Online 正确性:", "✅ PASSED" if ok else
      f"❌ FAILED  max_diff={torch.max(torch.abs(B_ref - B_online)).item():.6f}")


2026-05-31 14:23:31  [TileLang:tilelang.jit.kernel:INFO] (kernel.py:130): TileLang begins to compile kernel `tl_softmax_online_gfx11xx` with `out_idx=[-1]`


2026-05-31 14:23:33  [TileLang:tilelang.jit.kernel:INFO] (kernel.py:138): TileLang completes to compile kernel `tl_softmax_online_gfx11xx`
TileLang Online 正确性: ✅ PASSED



import matplotlib.pyplot as plt

WARMUP, REPEAT = 20, 200

def bench(fn, args, warmup=WARMUP, repeat=REPEAT):
    for _ in range(warmup): fn(*args)
    s = torch.cuda.Event(enable_timing=True)
    e = torch.cuda.Event(enable_timing=True)
    torch.cuda.synchronize(); s.record()
    for _ in range(repeat): fn(*args)
    e.record(); torch.cuda.synchronize()
    return s.elapsed_time(e) / repeat

bytes_3pass  = N * M * 4 * 5   # 3R + 2W
bytes_online = N * M * 4 * 3   # 2R + 1W

# ── 基线结果 ─────────────────────────────────────────────────────────────────
results = {
    "PyTorch":              (bench(ref_softmax,    [A]), bytes_3pass),
    "Triton (3-pass)":      (bench(triton_softmax, [A]), bytes_3pass),
    "TileLang 3-pass":      (bench(k,              [A]), bytes_3pass),
    "TileLang Online\n(BM=256)": (bench(k_online,  [A]), bytes_online),
}

# ── gfx1151 优化变体（由前面各 cell 填充）──────────────────────────────────────
if arch.startswith("gfx1151"):
    if 'sweep' in dir() and sweep and best_bm != 256:
        results[f"TileLang Online\n(BM={best_bm})"] = (sweep[best_bm][0], bytes_online)
    for label, (ms_v, _) in (results_opt.items() if 'results_opt' in dir() else {}.items()):
        results[f"TileLang\n{label}"] = (ms_v, bytes_online)

pt_ms = results["PyTorch"][0]
print(f"Softmax  (N={N}, M={M}, float32, {arch})")
print(f"{'内核':<28}  {'ms':>8}  {'TB/s':>8}  {'vs PyTorch':>10}")
print("-" * 60)
for name, (ms, bytes) in results.items():
    label = name.replace('\n', ' ')
    bw = bytes / ms / 1e9
    vs = f"{pt_ms / ms:+.1%}"
    print(f"{label:<28}  {ms:>8.4f}  {bw:>8.3f}  {vs:>10}")
print("-" * 60)

# ── 柱状图 ───────────────────────────────────────────────────────────────────
labels = list(results.keys())
values = [v[0] for v in results.values()]
n_bars = len(values)
palette = ["#4C72B0", "#55A868", "#C44E52", "#8B0000",
           "#DD8452", "#937860", "#DA8BC3", "#8C8C8C"][:n_bars]

fig, ax = plt.subplots(figsize=(max(10, n_bars * 1.6), 4))
x = range(n_bars)
bars = ax.bar(x, values, color=palette, width=0.6, edgecolor="white", linewidth=0.8)
ax.set_xticks(list(x)); ax.set_xticklabels(labels, fontsize=8.5)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + max(values) * 0.01,
            f"{val:.4f}", ha="center", va="bottom", fontsize=8)
ax.set_ylabel("延迟 (ms)")
ax.set_title(f"Online Softmax 优化对比  (N={N}, M={M}, float32, {arch})")
ax.set_ylim(0, max(values) * 1.25)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()


In [7]:

# ── 方向一：BLOCK_M 扫描（仅 gfx1151）────────────────────────────────────────
# tl_softmax_online_gfx1151 已将 BLOCK_M 声明为普通 int 参数，
# 只需用不同 BM 值重新 compile() 即可对比性能。

if arch.startswith("gfx1151"):
    WARMUP, REPEAT = 20, 200

    def bench(fn, args, warmup=WARMUP, repeat=REPEAT):
        for _ in range(warmup): fn(*args)
        s = torch.cuda.Event(enable_timing=True)
        e = torch.cuda.Event(enable_timing=True)
        torch.cuda.synchronize(); s.record()
        for _ in range(repeat): fn(*args)
        e.record(); torch.cuda.synchronize()
        return s.elapsed_time(e) / repeat

    bytes_online = N * M * 4 * 3   # 2R + 1W
    sweep = {}
    print(f"BLOCK_M 扫描 — threads=256, N={N}, M={M}")
    print(f"{'BLOCK_M':>8}  {'tile数':>6}  {'ms':>8}  {'TB/s':>8}  状态")
    print("-" * 52)
    for bm in [256, 512, 1024, 2048, 4096]:
        if M % bm != 0:
            print(f"{bm:>8}  {'—':>6}  {'—':>8}  {'—':>8}  M 无法整除")
            continue
        try:
            k_bm = tl_softmax_online_gfx1151.compile(N=N, M=M, BLOCK_M=bm)
            B_test = k_bm(A.clone())
            ok_bm = torch.allclose(B_ref, B_test, atol=5e-4)
            if not ok_bm:
                print(f"{bm:>8}  {M//bm:>6}  {'—':>8}  {'—':>8}  ❌ 数值错误")
                continue
            ms_bm = bench(k_bm, [A])
            bw_bm = bytes_online / ms_bm / 1e9
            sweep[bm] = (ms_bm, k_bm)
            marker = " ← 基线" if bm == 256 else ""
            print(f"{bm:>8}  {M//bm:>6}  {ms_bm:>8.4f}  {bw_bm:>8.3f}  ✅{marker}")
        except Exception as exc:
            print(f"{bm:>8}  {M//bm:>6}  {'—':>8}  {'—':>8}  编译失败: {type(exc).__name__}")

    if sweep:
        best_bm, (best_ms, best_k) = min(sweep.items(), key=lambda x: x[1][0])
        baseline_ms = sweep[256][0] if 256 in sweep else None
        speedup = f"{baseline_ms / best_ms:.2f}×" if baseline_ms else "?"
        print(f"\n最优: BLOCK_M={best_bm} → {best_ms:.4f} ms  ({speedup} vs BM=256)")
    else:
        best_bm, best_k, best_ms = 256, k_online, None
        print("所有配置均失败，保留基线 BM=256")
else:
    print(f"arch={arch}: BLOCK_M 扫描仅针对 gfx1151，跳过。")
    best_bm, best_k = BM_online, k_online


arch=gfx1100: BLOCK_M 扫描仅针对 gfx1151，跳过。


## gfx1151 优化 — 方向二：行合并

N=4096 行、20 CU 的条件下，基线已产生 4096 个 block（每 CU 约 204 个），占用率不是瓶颈。但将 **R 行顺序处理在同一个 block 内**，可以让 wavefront 在两行之间复用 L1/L2 缓存状态，并在一行串行 reduce 链结束时保持指令流水线繁忙，减少等待气泡。

这里取 `R=2`（每 block 处理 2 行，grid 缩为 N//2）。fragment 分配在两行间共享；只有标量累加器 `mx`/`sm` 在每行开始时重新初始化。

## gfx1151 优化 — 方向三：双缓冲预取

Online 串行循环每个 tile 的结构如下：

```
LOAD A[row, tile]          ← 全局内存延迟暴露在这里
reduce_max → 更新 mx
T.Parallel(exp)
reduce_sum → 更新 sm
```

分配两个乒乓 fragment，将 **tile i+1 的 load** 与 **tile i 的 reduce/compute** 重叠，即可隐藏全局内存延迟。RDNA3.5 的 `buffer_load` 指令在 wavefront 执行 ALU 时可以同时在途——双缓冲无需任何显式 async API 即可利用这一特性。

实现：两个 fragment `Al0`/`Al1` 每次迭代交替角色（`buf ^ 1` 选择空闲 buffer 用于加载，同时活跃 buffer 正在处理）。

In [8]:

# ── 方向二：2 行合并 ──────────────────────────────────────────────────────────
@tilelang.jit(pass_configs={
    tilelang.PassConfigKey.TL_DISABLE_WARP_SPECIALIZED: True,
    tilelang.PassConfigKey.TL_DISABLE_TMA_LOWER: True,
})
def tl_softmax_online_2rows(A, BLOCK_M: int):
    """在一个 kernel block 内顺序处理 2 行，fragment 复用，mx/sm 每行重新初始化。"""
    log2_e = 1.44269504
    N_, M_ = T.const("N, M")
    dtype  = T.float32
    A: T.Tensor((N_, M_), dtype)
    B = T.empty((N_, M_), dtype)
    with T.Kernel(N_ // 2, threads=256) as blk:
        Al    = T.alloc_fragment((1, BLOCK_M), dtype)
        mx    = T.alloc_fragment((1,), dtype)
        sm    = T.alloc_fragment((1,), dtype)
        mx_bl = T.alloc_fragment((1,), dtype)
        sm_bl = T.alloc_fragment((1,), dtype)
        for r in T.Serial(2):
            row = blk * 2 + r
            T.fill(mx, float("-inf"))
            T.clear(sm)
            for m_blk in T.Serial(M_ // BLOCK_M):
                T.copy(A[row, m_blk * BLOCK_M], Al)
                T.fill(mx_bl, float("-inf"))
                T.reduce_max(Al, mx_bl, dim=1, clear=True)
                for i in T.Parallel(1):
                    new_mx = T.max(mx[0], mx_bl[0])
                    sm[0]  = sm[0] * T.exp2(log2_e * (mx[0] - new_mx))
                    mx[0]  = new_mx
                for j in T.Parallel(BLOCK_M):
                    Al[0, j] = T.exp2(log2_e * (Al[0, j] - mx[0]))
                T.clear(sm_bl)
                T.reduce_sum(Al, sm_bl, dim=1, clear=True)
                for i in T.Parallel(1):
                    sm[0] = sm[0] + sm_bl[0]
            for m_blk in T.Serial(M_ // BLOCK_M):
                T.copy(A[row, m_blk * BLOCK_M], Al)
                for j in T.Parallel(BLOCK_M):
                    B[row, m_blk * BLOCK_M + j] = T.exp2(log2_e * (Al[0, j] - mx[0])) / sm[0]
    return B


# ── 方向三：双缓冲预取 ────────────────────────────────────────────────────────
@tilelang.jit(pass_configs={
    tilelang.PassConfigKey.TL_DISABLE_WARP_SPECIALIZED: True,
    tilelang.PassConfigKey.TL_DISABLE_TMA_LOWER: True,
})
def tl_softmax_online_dbuf(A, BLOCK_M: int):
    """乒乓两个 fragment：tile-i+1 的 load 与 tile-i 的 reduce/compute 重叠执行。
    编译器看到 reduce barrier 前的 load，可将 buffer_load 指令插入 ALU 空隙。"""
    log2_e   = 1.44269504
    N_, M_   = T.const("N, M")
    dtype    = T.float32
    A: T.Tensor((N_, M_), dtype)
    B = T.empty((N_, M_), dtype)
    n_tiles  = M_ // BLOCK_M
    with T.Kernel(N_, threads=256) as row:
        Al0   = T.alloc_fragment((1, BLOCK_M), dtype)   # ping
        Al1   = T.alloc_fragment((1, BLOCK_M), dtype)   # pong
        mx    = T.alloc_fragment((1,), dtype)
        sm    = T.alloc_fragment((1,), dtype)
        mx_bl = T.alloc_fragment((1,), dtype)
        sm_bl = T.alloc_fragment((1,), dtype)
        T.fill(mx, float("-inf"))
        T.clear(sm)
        # 序言：预取 tile 0 到 Al0
        T.copy(A[row, 0], Al0)
        for m_blk in T.Serial(n_tiles - 1):
            # 预取下一个 tile 到空闲 buffer
            for j in T.Parallel(BLOCK_M):
                Al1[0, j] = A[row, (m_blk + 1) * BLOCK_M + j]
            # 处理当前 tile（Al0）
            T.fill(mx_bl, float("-inf"))
            T.reduce_max(Al0, mx_bl, dim=1, clear=True)
            for i in T.Parallel(1):
                new_mx = T.max(mx[0], mx_bl[0])
                sm[0]  = sm[0] * T.exp2(log2_e * (mx[0] - new_mx))
                mx[0]  = new_mx
            for j in T.Parallel(BLOCK_M):
                Al0[0, j] = T.exp2(log2_e * (Al0[0, j] - mx[0]))
            T.clear(sm_bl)
            T.reduce_sum(Al0, sm_bl, dim=1, clear=True)
            for i in T.Parallel(1):
                sm[0] = sm[0] + sm_bl[0]
            # 交换：Al0 ← Al1，供下次迭代使用
            for j in T.Parallel(BLOCK_M):
                Al0[0, j] = Al1[0, j]
        # 结语：处理已在 Al0 中的最后一个 tile
        T.fill(mx_bl, float("-inf"))
        T.reduce_max(Al0, mx_bl, dim=1, clear=True)
        for i in T.Parallel(1):
            new_mx = T.max(mx[0], mx_bl[0])
            sm[0]  = sm[0] * T.exp2(log2_e * (mx[0] - new_mx))
            mx[0]  = new_mx
        for j in T.Parallel(BLOCK_M):
            Al0[0, j] = T.exp2(log2_e * (Al0[0, j] - mx[0]))
        T.clear(sm_bl)
        T.reduce_sum(Al0, sm_bl, dim=1, clear=True)
        for i in T.Parallel(1):
            sm[0] = sm[0] + sm_bl[0]
        # Pass 2：归一化写出
        for m_blk in T.Serial(n_tiles):
            T.copy(A[row, m_blk * BLOCK_M], Al0)
            for j in T.Parallel(BLOCK_M):
                B[row, m_blk * BLOCK_M + j] = T.exp2(log2_e * (Al0[0, j] - mx[0])) / sm[0]
    return B


# ── 编译 + 正确性快检 ─────────────────────────────────────────────────────────
if arch.startswith("gfx1151"):
    results_opt = {}

    for label, jit_fn in [("2行合并",  tl_softmax_online_2rows),
                           ("双缓冲",   tl_softmax_online_dbuf)]:
        try:
            k_v = jit_fn.compile(N=N, M=M, BLOCK_M=best_bm)
            B_v = k_v(A.clone())
            ok_v = torch.allclose(B_ref, B_v, atol=5e-4)
            ms_v = bench(k_v, [A])
            bw_v = bytes_online / ms_v / 1e9
            results_opt[label] = (ms_v, k_v)
            status = "✅" if ok_v else "❌ 数值错误"
            print(f"{label} (BM={best_bm}): {ms_v:.4f} ms  {bw_v:.3f} TB/s  {status}")
        except Exception as exc:
            print(f"{label}: 编译失败 — {type(exc).__name__}: {str(exc)[:100]}")
else:
    print(f"arch={arch}: 方向 2/3 仅针对 gfx1151，跳过。")
    results_opt = {}


arch=gfx1100: 方向 2/3 仅针对 gfx1151，跳过。


In [9]:
WARMUP, REPEAT = 20, 200

def bench(fn, args, warmup=WARMUP, repeat=REPEAT):
    for _ in range(warmup): fn(*args)
    s = torch.cuda.Event(enable_timing=True)
    e = torch.cuda.Event(enable_timing=True)
    torch.cuda.synchronize(); s.record()
    for _ in range(repeat): fn(*args)
    e.record(); torch.cuda.synchronize()
    return s.elapsed_time(e) / repeat

results_all = {
    "PyTorch":          bench(ref_softmax,        [A]),
    "Triton":           bench(triton_softmax,     [A]),
    "TileLang 3-pass":  bench(k,                 [A]),
    "TileLang Online":  bench(k_online,           [A]),
}

bytes_rw = N * M * 4 * 2
pt_ms  = results_all["PyTorch"]
tri_ms = results_all["Triton"]

header = f"{'实现':<20}  {'延迟 (ms)':>10}  {'带宽 (TB/s)':>12}  {'vs PyTorch':>10}  {'vs Triton':>10}"
sep    = "-" * len(header)
print(f"\nSoftmax 全对比  (N={N}, M={M}, float32)")
print(sep)
print(header)
print(sep)
for name, ms in results_all.items():
    bw     = bytes_rw / ms / 1e9
    vs_pt  = f"{pt_ms/ms:+.1%}"
    vs_tri = f"{tri_ms/ms:+.1%}"
    print(f"{name:<20}  {ms:>10.4f}  {bw:>12.2f}  {vs_pt:>10}  {vs_tri:>10}")
print(sep)



Softmax 全对比  (N=4096, M=16384, float32)
----------------------------------------------------------------------
实现                       延迟 (ms)     带宽 (TB/s)  vs PyTorch   vs Triton
----------------------------------------------------------------------
PyTorch                   0.7204          0.75     +100.0%     +119.3%
Triton                    0.8595          0.62      +83.8%     +100.0%
TileLang 3-pass           1.1604          0.46      +62.1%      +74.1%
TileLang Online           0.6483          0.83     +111.1%     +132.6%
----------------------------------------------------------------------
